# R3: Failure-case analysis — where do errors come from?

Reviewer 3 asks for a failure-case analysis and notes the claim "recall governs accuracy" is overstated. We classify every scoreable generation and split failures into **retrieval-attributable** (a gold column was not retrieved) vs **generation-attributable** (all gold columns retrieved, result still wrong).

In [1]:
# Locate the master results table (released with the repo).
import os
from pathlib import Path
import pandas as pd, numpy as np

CANDIDATES = [os.environ.get("MASTER_CSV", ""),
              "results/tables/generation_results.csv",
              "../results/tables/generation_results.csv",
              "notebooks/results/tables/generation_results.csv"]
MASTER = next((p for p in CANDIDATES if p and Path(p).exists()), None)
assert MASTER, "Set MASTER_CSV or place generation_results.csv under results/tables/"
df = pd.read_csv(MASTER)
print("rows:", len(df), "| conditions:", sorted(df.condition.unique()))
print("tiers:", sorted(df.model_tier.unique()), "| datasets:", sorted(df.dataset.unique()))
OUT = Path("results"); (OUT/"tables").mkdir(parents=True, exist_ok=True); (OUT/"figures").mkdir(parents=True, exist_ok=True)


rows: 32598 | conditions: ['C1', 'C2', 'C3', 'C4', 'C5_L1', 'C5_L2', 'C5_L3']
tiers: ['MID', 'STRONG', 'WEAK'] | datasets: ['D1', 'D2', 'D2_fin']


In [2]:
def classify(r):
    if r.valid_sql==0: return 'invalid_sql'
    if pd.isna(r.exec_match): return 'unscoreable'
    if r.exec_match==1: return 'correct'
    err=str(r.pred_error) if pd.notna(r.pred_error) else ''
    return 'exec_error' if err.strip() else 'wrong_result'
df['cat']=df.apply(classify,axis=1)
rows=[]
for tier in ['STRONG','MID','WEAK']:
    for ds in ['D1','D2']:
        sub=df[(df.model_tier==tier)&(df.dataset==ds)&(df.condition=='C4')]
        fail=sub[sub.cat.isin(['wrong_result','exec_error','invalid_sql'])]
        miss=fail[fail.col_recall_strict<0.999]
        rows.append({'dataset':ds,'tier':tier,'EX':round(sub.exec_match.mean(),3),'n_fail':len(fail),
            'retrieval_attrib_%':round(100*len(miss)/len(fail),1) if len(fail) else np.nan,
            'generation_attrib_%':round(100*(len(fail)-len(miss))/len(fail),1) if len(fail) else np.nan})
t=pd.DataFrame(rows); t.to_csv('results/tables/NEW_failure_taxonomy.csv',index=False)
print(t.to_string(index=False))

dataset   tier    EX  n_fail  retrieval_attrib_%  generation_attrib_%
     D1 STRONG 0.778     230                19.6                 80.4
     D2 STRONG 0.532     233                12.9                 87.1
     D1    MID 0.761     247                20.2                 79.8
     D2    MID 0.500     249                12.0                 88.0
     D1   WEAK 0.744     265                17.7                 82.3
     D2   WEAK 0.444     277                12.6                 87.4


In [3]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
st=t[t.tier=='STRONG']; x=np.arange(len(st))
fig,ax=plt.subplots(figsize=(5,3.2))
ax.bar(x,st['retrieval_attrib_%'],color='0.3',edgecolor='black',label='Retrieval-attributable (missing gold col)')
ax.bar(x,st['generation_attrib_%'],bottom=st['retrieval_attrib_%'],color='0.7',edgecolor='black',hatch='///',label='Generation-attributable (recall=1, still wrong)')
ax.set_xticks(x); ax.set_xticklabels([f'{r.dataset}' for _,r in st.iterrows()]); ax.set_ylabel('% of failures'); ax.set_ylim(0,100); ax.set_title('C4 failures (STRONG)'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig('results/figures/NEW_failure_taxonomy.png',dpi=150); plt.show()

**Takeaway:** at the hybrid operating point (C4, recall ≈ 0.97), 80–88% of failures retrieved every gold column and still failed — they are generation-attributable — stable across all tiers. Recall is therefore **necessary but not sufficient**: when degraded it binds accuracy, but once preserved the residual errors are generation-side. This is the evidence for the conditioned claim in the revised manuscript.